In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import glob
import os
import numpy as np
import h5py
import matplotlib.colors as colors
import random

import sys, os
# This is not super pretty, but I think this is the best way to import stuff from ../../../util?
CODE_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if CODE_ROOT not in sys.path:
    sys.path.insert(1, CODE_ROOT)

from util.sim_data_helpers import get_data_from_snap_folder, get_data_from_header, get_cosmo_parameters

In [ ]:
def neutral_hydrogen_hist(path, snapN, resolution, little_h, _range=None):

    # load gas data
    coordinates = get_data_from_snap_folder(path, snapN, "PartType0", "Coordinates")
    masses = get_data_from_snap_folder(path, snapN, "PartType0", "Masses") * 1e10/little_h  # M_sun
    f_HI = get_data_from_snap_folder(path, snapN, "PartType0", "NeutralHydrogenAbundance")

    # neutral hydrogen mass per cell
    X_H = 0.76 # TODO: is this correct??
    HI_mass = masses * X_H * f_HI

    box_size = get_data_from_header(path, snapN, "BoxSize")  # ckpc/h

    physical_bin_volume = ((box_size/1e3)/resolution)**2 * (box_size/1e3)

    if _range is None:
        _range = [[0, box_size], [0, box_size]]

    # mass-weighted histogram
    h, xedges, yedges = np.histogram2d(
        coordinates[:,0],
        coordinates[:,1],
        bins=resolution,
        range=_range,
        weights=HI_mass
    )

    # convert to projected density
    h = h / physical_bin_volume

    return h, xedges, yedges

In [ ]:
gp_numbers_to_use = [ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9,
                     10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
                     20, 21, 22, 23, 24, 25, 26, 27, 28, 29,
                     30, 31, 32, 33, 34, 35, 36, 37, 38, 39,
                     40, 41, 42, 43, 44, 45, 46, 47, 48, 49]  # number 34, 45, 48 only ran ultil z = 2
gp_indices = [i for i in range(len(gp_numbers_to_use))]
random.shuffle(gp_indices)  # randomize which boxes are shown
# gp_indices = gp_indices[-4:]

base_path = "/vera/u/jerbo/my_ptmp/L25n256_suite/"
snapN = 1

sort_param = []
for gp_index in gp_indices:
    gp = gp_numbers_to_use[gp_index]
    path = base_path + f"gridpoint{gp}/"

    Omega0, OmegaBaryon, OmegaLambda, HubbleParam = get_cosmo_parameters(path)
    sort_param.append((OmegaBaryon, gp_index))

_, sorted_indices = zip(*sorted(sort_param))

print(len(gp_numbers_to_use))

In [ ]:
n_sim = 47
total_sims = len(gp_numbers_to_use)

indices_to_use = []
for i in range(n_sim):
    indices_to_use.append(int(i*((total_sims-1)/(n_sim-1))))

print(indices_to_use)

sorted_indices_selected = []
for i in indices_to_use:
    sorted_indices_selected.append(sorted_indices[i])

print(sorted_indices_selected)
#np.random.shuffle(sorted_indices_selected)
print(sorted_indices_selected)
"""
shuffled_indices = []
for i in range(int(len(sorted_indices_selected)/2)):
    shuffled_indices.append(sorted_indices_selected[i])
    shuffled_indices.append(sorted_indices_selected[n_sim-1-i])

print(shuffled_indices)
sorted_indices_selected = shuffled_indices"""

In [ ]:
data = {}

for gp_index in sorted_indices_selected:
    gp = gp_numbers_to_use[gp_index]
    path = base_path + f"gridpoint{gp}/"

    Omega0, OmegaBaryon, OmegaLambda, HubbleParam = get_cosmo_parameters(path)
    h, xedges, yedges = neutral_hydrogen_hist(path, snapN, 512, little_h=HubbleParam)

    data[gp] = {
        "h": h,
        "xedges": xedges,
        "yedges": yedges,
        "Omega0": Omega0,
        "OmegaLambda": OmegaLambda,
        "HubbleParam": HubbleParam,
        "Omegab": OmegaBaryon
    }

# reference normalization
gp_norm = gp_numbers_to_use[sorted_indices_selected[-1]]
norm_h = data[gp_norm]["h"]
vmin = norm_h[norm_h != 0].min()
vmax = norm_h.max()

In [ ]:
combined_images = []

for i in sorted_indices_selected:

    tiles = data[gp_numbers_to_use[i]]["h"]

    combined = np.vstack((
        np.hstack((tiles, tiles)),
        np.hstack((tiles, tiles))
    ))

    combined_images.append(combined)

In [ ]:
h, w = combined_images[0].shape
X, Y = np.indices((h, w))

offset = 0
slope = 0

stripe_width = w/(n_sim-slope)  # adjust this
mask_indices = ((X + Y*(slope*stripe_width/h)) // stripe_width) % len(combined_images) + offset

In [ ]:
final_image = np.zeros_like(combined_images[0])

for i, img in enumerate(combined_images):
    final_image[mask_indices == i] = img[mask_indices == i]

    #edges = np.gradient(mask_indices.astype(float))
    #boundary = (np.abs(edges[0]) + np.abs(edges[1])) > 0

    #final_image[boundary] = vmax  # or vmax → bright lines

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8), dpi=600)

norm = colors.LogNorm(vmin=vmin, vmax=vmax)

im = ax.imshow(
    final_image.T,
    origin="lower",
    cmap="magma",
    norm=norm,
    interpolation="None"
)

# number of stripes to cover the image
n_lines = n_sim + 2

for k in range(n_lines):
    c = k * stripe_width

    y_vals = np.linspace(0, h, 1000)
    x_vals = (c - y_vals*(slope*stripe_width/h))

    mask = (x_vals >= 0) & (x_vals <= w)

    if np.any(mask):
        ax.plot(
            x_vals[mask],
            y_vals[mask],
            color="black",
            linestyle="--",
            linewidth=1,
            alpha=0
        )

ax.set_xticks([])
ax.set_yticks([])
ax.set_xlim([0, w])
ax.set_ylim([0, h])
ax.set_facecolor("white")

plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
plt.savefig("title_plots/test4.pdf", format="PDF")
plt.show()